In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , RobustScaler , OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ( accuracy_score, 
                            precision_score, 
                            recall_score,
                            f1_score,balanced_accuracy_score,
                            confusion_matrix,roc_curve,
                            roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV
df= pd.read_csv("data.csv")

1 | Data Integrity Fixes & Pre-Split Engineering

In [2]:
# Convert empty string spaces to true numerical NaNs
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

# Cast the entire column to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df.drop(['customerID'], axis = 1,inplace=True)
df['Churn'] = df['Churn'].replace({'No': 0, 'Yes': 1})

In [3]:
X = df.drop(columns = ['Churn'])
y=df['Churn']
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,stratify=y)

2 | Automated Model Training & Hyperparameter Tuning

In [4]:
# ==============================================================================
# 1. THE CUSTOM FEATURE ENGINEERING CLASS
# ==============================================================================
class ChurnFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, tenure_col='tenure', monthly_col='MonthlyCharges'):
        self.tenure_col = tenure_col
        self.monthly_col = monthly_col
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_out = X.copy()
        # Securely append the interaction ratio while keeping parent main effects intact
        X_out['Charges_to_Tenure_Ratio'] = X_out[self.monthly_col] / (X_out[self.tenure_col] + 1)
        return X_out

In [5]:
# 1. Define your initial base columns
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols_ohe = ['PaymentMethod', 'Contract', 'InternetService']

# 2. Add the engineered ratio name to the numeric list BEFORE subtracting from the column pool
num_cols_extended = num_cols + ['Charges_to_Tenure_Ratio']

# 3. Safely deduce the remaining label encoded categorical columns
cat_cols_le = list(set(X_train.columns) - set(num_cols_extended) - set(cat_cols_ohe))

# ==============================================================================
# RE-DESIGNED PREPROCESSOR (With Imputer Inside)
# ==============================================================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Catches TotalCharges and Ratio NaNs!
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols_extended),
        ('cat_ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols_ohe),
        ('cat_le', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols_le)
    ]
)

In [6]:
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [8]:
from sklearn.svm import SVC

In [15]:

# NON-LINEAR SVM CHAMPION PIPELINE
svm_pipeline = Pipeline(steps=[
    ("feature_engineer", ChurnFeatureEngineer(tenure_col='tenure', monthly_col='MonthlyCharges')),
    ("preprocessor", preprocessor), 
    # probability=True is required so we can perform threshold tuning later
    ("model", SVC(class_weight='balanced', probability=True, random_state=42))
])

# Regularization (C) and Kernel flexibility (gamma) grid space
param_grid = {
    'model__C': [0.1, 1.0, 10.0],
    'model__gamma': ['scale', 'auto', 0.01],
    'model__kernel': ['rbf'] # Radial Basis Function maps the non-linear boundaries
}

print("⚡ Training Support Vector Machine with Kernel Trick (This may take a minute)...")
grid_search_svm = GridSearchCV(svm_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search_svm.fit(X_train, y_train)

final_svm_model = grid_search_svm.best_estimator_
print(f"✅ Best SVM Parameters: {grid_search_svm.best_params_}\n")

⚡ Training Support Vector Machine with Kernel Trick (This may take a minute)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits


ValueError: 
All the 45 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
45 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\elitebook\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elitebook\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\elitebook\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^

In [11]:
# NON-LINEAR SVM CHAMPION PIPELINE

svm_pipeline = Pipeline(steps=[
    ("feature_engineer", ChurnFeatureEngineer(tenure_col='tenure', monthly_col='MonthlyCharges')),
    ("preprocessor", preprocessor), 
    # probability=True is required so we can perform threshold tuning later
    ("model", SVC(class_weight='balanced', probability=True, random_state=42))
])

# Regularization (C) and Kernel flexibility (gamma) grid space
param_grid = {
    'model__C': [0.1, 1.0, 10.0],
    'model__gamma': ['scale', 'auto', 0.01],
    'model__kernel': ['rbf'] # Radial Basis Function maps the non-linear boundaries
}

print("⚡ Training Support Vector Machine with Kernel Trick (This may take a minute)...")
grid_search_svm = GridSearchCV(svm_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search_svm.fit(X_train, y_train)

final_svm_model = grid_search_svm.best_estimator_
print(f"✅ Best SVM Parameters: {grid_search_svm.best_params_}\n")


⚡ Training Support Vector Machine with Kernel Trick (This may take a minute)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits


ValueError: 
All the 45 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
45 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\elitebook\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elitebook\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\elitebook\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^

In [12]:
# POST-PROCESSING THRESHOLD OPTIMIZATION

y_proba_svm = final_svm_model.predict_proba(X_test)[:, 1]
precisions_svm, recalls_svm, thresholds_svm = precision_recall_curve(y_test, y_proba_svm)
f1_scores_svm = 2 * (precisions_svm * recalls_svm) / (precisions_svm + recalls_svm + 1e-10)
best_threshold_svm = thresholds_svm[np.argmax(f1_scores_svm[:-1])]
print(f"🥇 Optimal SVM Threshold: {best_threshold_svm:.3f}\n")



NameError: name 'final_svm_model' is not defined

In [14]:
# PERFORMANCE RECORD & HEATMAP

y_pred_svm = (y_proba_svm >= best_threshold_svm).astype(int)

print("📝 Final Tuned SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

plt.figure(figsize=(5, 4))
cm_svm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt="d", cmap="RdPu", cbar=False,
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.title(f"Tuned SVM (Threshold = {best_threshold_svm:.2f})", fontweight="bold")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

NameError: name 'y_proba_svm' is not defined